# Агент и оценка по эталонам

Запуск в Google Colab. Добавьте ключ OpenRouter в Colab Secrets под именем `OPENROUTER_API_KEY`.

In [ ]:
!git clone --quiet https://github.com/klinich-1/Analysis-of-structured-data.git
%cd Analysis-of-structured-data
!pip install --quiet -r agent/requirements.txt
!python -c "import zipfile; zipfile.ZipFile('data/parking_team_package_no_pdf.zip').extractall()"

In [ ]:
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')

## Один вопрос

In [ ]:
!python agent/csv_agent.py parking_team_package/data/parking_places_hourly_202605.csv "Сколько строк с freeParking = -1?"

## Эксперимент

Сначала проверьте три вопроса. Измените `MODEL`, чтобы сравнить другую модель.

In [ ]:
MODEL = 'anthropic/claude-opus-5-5'
!python agent/evaluate.py questions --mode agent --condition protocol --model {MODEL} --limit 3
!python agent/evaluate.py summary

Полный прогон: 85 вопросов на каждое сочетание модели, режима и условия. Запуск можно повторить: готовые ответы пропускаются.

In [ ]:
for mode in ['agent', 'llm']:
    for condition in ['package', 'protocol']:
        !python agent/evaluate.py questions --mode {mode} --condition {condition} --model {MODEL}
!python agent/evaluate.py summary

## Проверка качества данных

Два человека независимо заполняют `results/defect_annotation.csv`.

In [ ]:
!python agent/evaluate.py defects --model {MODEL}

## Сохранение результатов

Colab удаляет файлы после отключения, поэтому скачайте результаты.

In [ ]:
from google.colab import files

!cd results && zip -q ../results.zip *
files.download('results.zip')